In [ ]:
# tx%2BB1WlWMfZSnZjtDeggmc%2BipIJcgeWAwHQN8%2Bd3KE3ZGlna5q3TrCbnplkMfAcKrGNfBuj0pHDfb%2Bh68nED%2Fg%3D%3D

In [ ]:
import pandas as pd

df = pd.read_csv("2025_06_07.csv", encoding="cp949", sep="\t")
print(df)

      □ 본 서비스에서 제공하는 정보는 법적인 효력이 없으므로 참고용으로만 활용하시기 바랍니다.
0      □ 신고정보가 실시간 변경, 해제되어 제공시점에 따라 공개건수 및 내용이 상이할 수...
1      □ 본 자료는 계약일 기준입니다. (※ 7월 계약, 8월 신고건 → 7월 거래건으로...
2      □ 통계자료 활용시에는 수치가 왜곡될 수 있으니 참고자료로만 활용하시기  바라며, ...
3                                                    NaN
4      * 국토교통부 실거래가 공개시스템의 궁금하신 점이나 문의사항은 콜센터 1533-29...
...                                                  ...
23994  23980,"경상남도 김해시 삼계동","1***","12m미만","40.00","월...
23995  23981,"충청남도 논산시 내동","1***","25m이상","33.00","월세...
23996  23982,"경기도 시흥시 대야동","5**","12m미만","49.58","전세"...
23997  23983,"전라남도 여수시 문수동","2**","12m미만","32.40","월세...
23998  23984,"부산광역시 금정구 금사동","2*","8m미만","93.70","월세"...

[23999 rows x 1 columns]


In [16]:
df = pd.read_csv("2025_06_07.csv", encoding="cp949", sep="\t", skiprows=15)

pd.set_option('display.max_columns', None)  # 모든 열 출력
pd.set_option('display.width', 1000)        # 가로 폭 넓게
pd.set_option('display.max_colwidth', None) # 문자열 길이 제한 해제

print(df)

      NO,"시군구","번지","도로조건","계약면적(㎡)","전월세구분","계약년월","계약일","보증금(만원)","월세금(만원)","건축년도","도로명","계약기간","계약구분","갱신요구권 사용","종전계약 보증금(만원)","종전계약 월세(만원)","주택유형"
0                                  1,"경기도 성남시 분당구 정자동","4*","25m미만","50.00","월세","202507","05","8,000","50","1994","백현로144번길","-","-","-","","","단독다가구"
1                                   2,"경기도 수원시 팔달구 우만동","5*","8m미만","20.00","전세","202507","05","6,000","0","2002","월드컵로211번길","-","-","-","","","단독다가구"
2                                   3,"경기도 수원시 팔달구 인계동","1***","8m미만","21.00","월세","202507","05","500","49","2014","세지로152번길","-","-","-","","","단독다가구"
3                                 4,"경기도 성남시 수정구 태평동","3***","8m미만","25.00","월세","202507","05","1,000","45","1989","수정북로19번길","-","-","-","","","단독다가구"
4                                         5,"경기도 오산시 내삼미동","8**","12m미만","55.00","전세","202507","05","13,000","0","2014","수청로","-","-","-","","","단독다가구"
...                                                                                     

In [6]:
# !pip install selenium webdriver-manager
!pip install webdriver-manager

In [20]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from datetime import datetime, timedelta
import time
import os


# 설정
download_dir = os.path.join(os.getcwd(), "real_estate_data")
os.makedirs(download_dir, exist_ok=True)

# Chrome 옵션 설정
chrome_options = webdriver.ChromeOptions()
prefs = {
    "download.default_directory": download_dir,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
}
chrome_options.add_experimental_option("prefs", prefs)

# 드라이버 설정
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)
driver.maximize_window()

def select_housing_type(housing_type):
    """주택 유형 선택 함수"""
    type_map = {
        '연립/다세대': 'xlsTab2',
        '단독/다가구': 'xlsTab3',
        '오피스텔': 'xlsTab4'
    }
    element_id = type_map[housing_type]
    WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.ID, element_id))
    ).click()
    print(f"{housing_type} 선택 완료")

def click_rent_button():
    """전월세 버튼 클릭 - 확실한 방법"""
    try:
        # li 태그의 클래스가 'on'인지 확인 (활성화 상태)
        rent_li = WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.XPATH, "//li[@class='list thing_lr on']/a[@title='전월세']"))
        )
        driver.execute_script("arguments[0].click();", rent_li)
        print("전월세 버튼 클릭 완료 (JavaScript)")
    except:
        # 만약 활성화되지 않았다면 직접 클릭 시도
        rent_btn = WebDriverWait(driver, 20).until(
            EC.element_to_be_clickable((By.XPATH, "//a[@title='전월세' and contains(@onclick, 'fnRtToLr')]"))
        )
        rent_btn.click()
        print("전월세 버튼 클릭 완료 (직접 클릭)")
    time.sleep(2)  # 로딩 대기

def set_date_range(start_date, end_date):
    """날짜 범위 설정"""
    # 시작일 설정
    start_input = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, "srhFromDt"))
    )
    start_input.clear()
    start_input.send_keys(start_date.strftime("%Y-%m-%d"))
    
    # 종료일 설정
    end_input = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, "srhToDt"))
    )
    end_input.clear()
    end_input.send_keys(end_date.strftime("%Y-%m-%d"))
    
    print(f"날짜 설정 완료: {start_date.strftime('%Y-%m-%d')} ~ {end_date.strftime('%Y-%m-%d')}")

def download_csv():
    """CSV 다운로드"""
    WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, "//button[contains(@onclick, 'fnCSVDown')]"))
    ).click()
    print("CSV 다운로드 시작")
    time.sleep(5)  # 다운로드 완료 대기

def main():
    try:
        # 웹사이트 접속
        driver.get("https://rt.molit.go.kr/pt/xls/xls.do?mobileAt=")
        time.sleep(3)  # 초기 로딩 대기

        # 주택 유형 리스트
        housing_types = ['연립/다세대', '단독/다가구', '오피스텔']
        
        # 날짜 범위 설정 (2024-07-10 ~ 2025-07-09)
        start_date = datetime(2024, 7, 10)
        end_date = datetime(2025, 7, 9)
        
        current_date = start_date
        
        while current_date <= end_date:
            # 해당 월의 첫날과 마지막 날 계산
            from_date = current_date.replace(day=1)
            to_date = (current_date.replace(day=28) + timedelta(days=4)).replace(day=1) - timedelta(days=1)
            
            # 종료일이 전체 종료일을 넘지 않도록 조정
            if to_date > end_date:
                to_date = end_date
            
            for housing_type in housing_types:
                print(f"\n[{housing_type}] {from_date.strftime('%Y-%m')} 데이터 처리 중...")
                
                # 주택 유형 선택
                select_housing_type(housing_type)
                time.sleep(1)
                
                # 전월세 선택
                set_contract_type()
                time.sleep(1)
                
                # 날짜 설정
                set_date_range(from_date, to_date)
                time.sleep(1)
                
                # CSV 다운로드
                download_csv()
                time.sleep(3)  # 서버 부하 방지를 위한 대기
            
            # 다음 달로 이동
            current_date = (current_date.replace(day=28) + timedelta(days=4)).replace(day=1)
            
    except Exception as e:
        print(f"오류 발생: {str(e)}")
        driver.save_screenshot("error.png")
    finally:
        driver.quit()
        print("모든 작업 완료")

if __name__ == "__main__":
    main()


[연립/다세대] 2024-07 데이터 처리 중...
연립/다세대 선택 완료
전월세 선택 완료 (방법2)
날짜 설정 완료: 2024-07-01 ~ 2024-07-31
CSV 다운로드 시작

[단독/다가구] 2024-07 데이터 처리 중...
단독/다가구 선택 완료
전월세 선택 완료 (방법2)
날짜 설정 완료: 2024-07-01 ~ 2024-07-31
CSV 다운로드 시작

[오피스텔] 2024-07 데이터 처리 중...
오피스텔 선택 완료
전월세 선택 완료 (방법2)
날짜 설정 완료: 2024-07-01 ~ 2024-07-31
CSV 다운로드 시작

[연립/다세대] 2024-08 데이터 처리 중...
연립/다세대 선택 완료
전월세 선택 완료 (방법2)
날짜 설정 완료: 2024-08-01 ~ 2024-08-31
CSV 다운로드 시작

[단독/다가구] 2024-08 데이터 처리 중...
단독/다가구 선택 완료
전월세 선택 완료 (방법2)
날짜 설정 완료: 2024-08-01 ~ 2024-08-31
CSV 다운로드 시작

[오피스텔] 2024-08 데이터 처리 중...
오피스텔 선택 완료
전월세 선택 완료 (방법2)
날짜 설정 완료: 2024-08-01 ~ 2024-08-31
CSV 다운로드 시작

[연립/다세대] 2024-09 데이터 처리 중...
연립/다세대 선택 완료
전월세 선택 완료 (방법2)
날짜 설정 완료: 2024-09-01 ~ 2024-09-30
CSV 다운로드 시작

[단독/다가구] 2024-09 데이터 처리 중...
단독/다가구 선택 완료
전월세 선택 완료 (방법2)
날짜 설정 완료: 2024-09-01 ~ 2024-09-30
CSV 다운로드 시작

[오피스텔] 2024-09 데이터 처리 중...
오피스텔 선택 완료
전월세 선택 완료 (방법2)
날짜 설정 완료: 2024-09-01 ~ 2024-09-30
CSV 다운로드 시작

[연립/다세대] 2024-10 데이터 처리 중...
연립/다세대 선택 완료
전월세 선택 완료 (방법2)
날짜 설정 완료

In [56]:
import os
import pandas as pd

# 원본 파일들이 있는 디렉토리 경로
input_dir = 'real_estate_data'
# 처리된 파일을 저장할 디렉토리 경로
output_dir = 'real_estate_data/cleaned'
os.makedirs(output_dir, exist_ok=True)

categories = ["단독다가구", "연립다세대", "오피스텔"]
encoding_type = "cp949"

for category in categories:
    matching_files = [f for f in os.listdir(input_dir) if category in f and f.endswith('.csv')]
    
    if not matching_files:
        print(f"[경고] '{category}'가 포함된 CSV 파일이 없습니다.")
        continue
    
    # 첫 번째 파일로 헤더 확인 (CP949/EUC-KR 인코딩으로 읽기)
    try:
        first_file = pd.read_csv(
            os.path.join(input_dir, matching_files[0]), 
            nrows=1, 
            encoding=encoding_type
        )
    except UnicodeDecodeError:
        # CP949도 실패하면 UTF-8 시도 (필요시)
        first_file = pd.read_csv(
            os.path.join(input_dir, matching_files[0]), 
            nrows=1, 
            encoding="utf-8"
        )
    
    # 헤더 확인 후 skiprows 결정
    if "NO" not in first_file.columns:  # 헤더가 없는 경우 (예: 설명글로 시작)
        merged_df = pd.concat(
            [pd.read_csv(
                os.path.join(input_dir, f), 
                skiprows=15, 
                encoding=encoding_type  # 명시적 인코딩 지정
            ) for f in matching_files],
            ignore_index=True
        )
    else:  # 헤더가 있는 경우
        merged_df = pd.concat(
            [pd.read_csv(
                os.path.join(input_dir, f), 
                encoding=encoding_type  # 명시적 인코딩 지정
            ) for f in matching_files],
            ignore_index=True
        )
    
    # 저장 시에도 UTF-8-SIG로 인코딩 (한글 깨짐 방지)
    output_filename = f"merged_{category}.csv"
    output_path = os.path.join(output_dir, output_filename)
    merged_df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"✅ '{category}' 파일 {len(matching_files)}개를 {output_filename}으로 합쳤습니다.")

print("\n모든 파일 병합 완료!")

C:\Users\Playdata\AppData\Local\Temp\ipykernel_19560\3385682861.py:38: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(
C:\Users\Playdata\AppData\Local\Temp\ipykernel_19560\3385682861.py:38: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(
C:\Users\Playdata\AppData\Local\Temp\ipykernel_19560\3385682861.py:38: DtypeWarning: Columns (9,16) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(
C:\Users\Playdata\AppData\Local\Temp\ipykernel_19560\3385682861.py:38: DtypeWarning: Columns (9,16) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(
C:\Users\Playdata\AppData\Local\Temp\ipykernel_19560\3385682861.py:38: DtypeWarning: Columns (9,16) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(


✅ '단독다가구' 파일 13개를 merged_단독다가구.csv으로 합쳤습니다.
✅ '연립다세대' 파일 13개를 merged_연립다세대.csv으로 합쳤습니다.
✅ '오피스텔' 파일 13개를 merged_오피스텔.csv으로 합쳤습니다.

모든 파일 병합 완료!


In [ ]:
import os
import pandas as pd

# 원본 파일들이 있는 디렉토리 경로
input_dir = 'real_estate_data/cleaned'
# 처리된 파일을 저장할 디렉토리 경로
output_dir = 'real_estate_data/cleaned'

# 출력 디렉토리 생성 (없을 경우)
os.makedirs(output_dir, exist_ok=True)

for filename in os.listdir(input_dir):
    if filename.startswith('merged'):
        filepath = os.path.join(input_dir, filename)
        
        # 인코딩 시도: 먼저 cp949로 시도, 실패하면 utf-8 시도
        try:
            df = pd.read_csv(filepath, encoding='cp949')
        except UnicodeDecodeError:
            try:
                df = pd.read_csv(filepath, encoding='utf-8')
            except UnicodeDecodeError:
                print(f"⚠️ {filename} 파일의 인코딩을 확인해주세요 (cp949/utf-8 모두 실패)")
                continue
        
        # '시군구' 컬럼에서 '서울특별시'와 '광역시'가 포함되지 않은 행만 선택
        if '시군구' in df.columns:
            df = df[~df['시군구'].str.contains('서울특별시', na=False)]
            df = df[~df['시군구'].str.contains('광역시', na=False)]
        else:
            print(f"⚠️ {filename} 파일에 '시군구' 컬럼이 없습니다.")
            continue
        
        # '전월세구분' 컬럼에서 '전세'가 포함되지 않은 행만 선택
        if '전월세구분' in df.columns:
            df = df[~df['전월세구분'].str.contains('전세', na=False)]
        else:
            print(f"⚠️ {filename} 파일에 '전월세구분' 컬럼이 없습니다.")
            continue
        
        # '전월세구분' 컬럼에서 '전세'가 포함되지 않은 행만 선택
        if '계약면적(㎡)' in df.columns:
            df = df[df['계약면적(㎡)'] < 66]
        elif '전용면적(㎡)' in df.columns:
            df = df[df['전용면적(㎡)'] < 60]
        else:
            print("\n⚠️ '전용면적' 컬럼이 존재하지 않아 면적 필터링을 건너뜁니다.")

        new_filename = 'cleaned_' + filename
        output_path = os.path.join(output_dir, new_filename)
        df.to_csv(output_path, index=False, encoding='utf-8-sig')
        
        print(f'{filename} 처리 완료 -> {new_filename}')

print("\n모든 파일 처리 완료!")

C:\Users\Playdata\AppData\Local\Temp\ipykernel_19560\2327061075.py:21: DtypeWarning: Columns (9,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath, encoding='utf-8')


merged_단독다가구.csv 처리 완료 -> cleaned_merged_단독다가구.csv


C:\Users\Playdata\AppData\Local\Temp\ipykernel_19560\2327061075.py:21: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath, encoding='utf-8')


merged_연립다세대.csv 처리 완료 -> cleaned_merged_연립다세대.csv


C:\Users\Playdata\AppData\Local\Temp\ipykernel_19560\2327061075.py:21: DtypeWarning: Columns (11,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath, encoding='utf-8')


merged_오피스텔.csv 처리 완료 -> cleaned_merged_오피스텔.csv

모든 파일 처리 완료!


In [12]:
def clean_monthly_rent(rent):
    # 1. 결측치 처리
    if pd.isna(rent):
        return np.nan
    
    # 2. 숫자 타입인 경우 정수로 변환
    if isinstance(rent, (int, float)):
        return int(rent)
    
    # 3. 문자열 처리
    if isinstance(rent, str):
        rent = rent.strip()  # 공백 제거
        rent = rent.replace(',', '')  # 쉼표 제거 (예: "50,000" → "50000")
        
        # 3-1. 숫자로 변환 가능한 문자열인지 확인
        if rent.replace('.', '', 1).isdigit():  # 소수점 포함 숫자 체크
            return int(float(rent))  # "630.0" → 630
        elif rent.isdigit():  # 정수형 문자열
            return int(rent)

# 파일 처리 함수
def process_files(file_list):
    for file in file_list:
        try:
            # 파일 읽기
            try:
                df = pd.read_csv(file, encoding='cp949')
            except UnicodeDecodeError:
                df = pd.read_csv(file, encoding='utf-8')
            
            print(f"\n{'='*50}")
            print(f"📊 파일명: {file}")
            
            # 월세금 컬럼 처리
            if '월세금(만원)' in df.columns:
                # 변환 전 데이터 샘플 출력
                print("\n🔍 변환 전 월세금 샘플:")
                print(df['월세금(만원)'].head(10))
                
                # 데이터 정제
                df['월세금(만원)'] = df['월세금(만원)'].apply(clean_monthly_rent)
                df['보증금(만원)'] = df['보증금(만원)'].apply(clean_monthly_rent)

                # 변환 후 데이터 샘플 출력
                print("\n✅ 변환 후 월세금 샘플:")
                print(df['월세금(만원)'].head(10))
                
                # 변환 결과 통계
                print("\n📊 변환 결과:")
                print(f"- NaN 개수: {df['월세금(만원)'].isna().sum()}")
                print(f"- 평균 월세금: {df['월세금(만원)'].mean():.1f} 만원")
                print(f"- 최대 월세금: {df['월세금(만원)'].max()} 만원")
            
                # 파일 저장
                new_filename = f"{file}_fin.csv"
                df.to_csv(new_filename, index=False, encoding='utf-8-sig')
                print(f"\n💾 파일 저장 완료: {new_filename}")
            else:
                print("⚠️ '월세금(만원)' 컬럼이 존재하지 않습니다.")
                
        except Exception as e:
            print(f"\n❌ {file} 처리 중 오류 발생: {str(e)}")

# 실행
file_list = [
    'real_estate_data/cleaned/cleaned_merged_단독다가구.csv',
    'real_estate_data/cleaned/cleaned_merged_연립다세대.csv', 
    'real_estate_data/cleaned/cleaned_merged_오피스텔.csv'
]

process_files(file_list)
print("\n모든 파일 처리 완료!")


📊 파일명: real_estate_data/cleaned/cleaned_merged_단독다가구.csv

🔍 변환 전 월세금 샘플:
0    52
1    64
2    27
3    36
4    55
5    50
6    45
7    30
8    34
9    10
Name: 월세금(만원), dtype: int64

✅ 변환 후 월세금 샘플:
0    52
1    64
2    27
3    36
4    55
5    50
6    45
7    30
8    34
9    10
Name: 월세금(만원), dtype: int64

📊 변환 결과:
- NaN 개수: 0
- 평균 월세금: 42.7 만원
- 최대 월세금: 400 만원

💾 파일 저장 완료: real_estate_data/cleaned/cleaned_merged_단독다가구.csv_fin.csv

📊 파일명: real_estate_data/cleaned/cleaned_merged_연립다세대.csv

🔍 변환 전 월세금 샘플:
0    15
1    40
2    40
3    35
4    65
5    40
6    19
7    50
8    15
9    50
Name: 월세금(만원), dtype: object


C:\Users\Playdata\AppData\Local\Temp\ipykernel_23040\62617871.py:29: DtypeWarning: Columns (11,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, encoding='utf-8')



✅ 변환 후 월세금 샘플:
0    15
1    40
2    40
3    35
4    65
5    40
6    19
7    50
8    15
9    50
Name: 월세금(만원), dtype: int64

📊 변환 결과:
- NaN 개수: 0
- 평균 월세금: 44.6 만원
- 최대 월세금: 1300 만원

💾 파일 저장 완료: real_estate_data/cleaned/cleaned_merged_연립다세대.csv_fin.csv


C:\Users\Playdata\AppData\Local\Temp\ipykernel_23040\62617871.py:29: DtypeWarning: Columns (11,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, encoding='utf-8')



📊 파일명: real_estate_data/cleaned/cleaned_merged_오피스텔.csv

🔍 변환 전 월세금 샘플:
0     18
1     35
2     26
3    103
4     40
5     40
6     55
7     34
8     37
9     45
Name: 월세금(만원), dtype: object

✅ 변환 후 월세금 샘플:
0     18
1     35
2     26
3    103
4     40
5     40
6     55
7     34
8     37
9     45
Name: 월세금(만원), dtype: int64

📊 변환 결과:
- NaN 개수: 0
- 평균 월세금: 53.6 만원
- 최대 월세금: 1300 만원

💾 파일 저장 완료: real_estate_data/cleaned/cleaned_merged_오피스텔.csv_fin.csv

모든 파일 처리 완료!


In [5]:
import pandas as pd
import numpy as np

# 파일 목록
file_list = [
    'real_estate_data/cleaned/cleaned_merged_단독다가구_fin.csv',
    'real_estate_data/cleaned/cleaned_merged_연립다세대_fin.csv', 
    'real_estate_data/cleaned/cleaned_merged_오피스텔_fin.csv'
]

for file in file_list:
    try:
        # 파일 읽기 (인코딩 시도: cp949 → utf-8)
        try:
            df = pd.read_csv(file, encoding='cp949')
        except UnicodeDecodeError:
            df = pd.read_csv(file, encoding='utf-8')
        
        print(f"\n{'='*50}")
        print(f"📊 파일명: {file}")
        print(f"📌 전체 행/열 수: {df.shape}")
        print(f"🔍 상위 5개 행:")
        print(df.head())
        print("\n📋 컬럼 정보:")
        print(df.info())
        print("\n🧮 수치형 데이터 요약:")
        print(df.describe())
        print(f"🔍 '월세금(만원)' unique 값")
        list=sorted(df['월세금(만원)'].unique())
        print(np.array(list).astype(int))  # 정렬하여 출력
        
    except FileNotFoundError:
        print(f"\n⚠️ {file} 파일을 찾을 수 없습니다.")
    except Exception as e:
        print(f"\n❌ {file} 처리 중 오류 발생: {str(e)}")

print("\n모든 파일 분석 완료!")


📊 파일명: real_estate_data/cleaned/cleaned_merged_단독다가구_fin.csv
📌 전체 행/열 수: (155138, 18)
🔍 상위 5개 행:
   NO                 시군구    번지   도로조건  계약면적(㎡) 전월세구분    계약년월  계약일  보증금(만원)  \
0   1  경상북도 포항시 남구 오천읍 원리   9**  12m미만     30.0    월세  202407   31      200   
1   2  경상북도 포항시 남구 오천읍 원리   9**  12m미만     40.0    월세  202407   31      200   
2   4    충청북도 청주시 청원구 우암동   3**  12m미만     25.0    월세  202407   31      100   
3  11  경상북도 포항시 남구 오천읍 원리   9**  12m미만     45.0    월세  202407   31      500   
4  20    충청북도 청주시 청원구 율량동  2***  12m미만     39.6    월세  202407   31      500   

   월세금(만원)    건축년도       도로명           계약기간 계약구분 갱신요구권 사용 종전계약 보증금(만원)  \
0       52     NaN  냉천로270번길  202408~202502   신규        -          NaN   
1       64  2024.0  냉천로270번길  202408~202502   신규        -          NaN   
2       27  2001.0   향군로15번길  202408~202507   신규        -          NaN   
3       36  2012.0  냉천로270번길  202408~202507   신규        -          NaN   
4       55  2018.0   율중로75번길  202407~202507   신규        -

C:\Users\Playdata\AppData\Local\Temp\ipykernel_27624\1947911892.py:17: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, encoding='utf-8')
C:\Users\Playdata\AppData\Local\Temp\ipykernel_27624\1947911892.py:17: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, encoding='utf-8')



📊 파일명: real_estate_data/cleaned/cleaned_merged_연립다세대_fin.csv
📌 전체 행/열 수: (37362, 21)
🔍 상위 5개 행:
   NO              시군구       번지    본번  부번        건물명 전월세구분  전용면적(㎡)    계약년월  \
0  18  경기도 수원시 권선구 세류동    545-3   545   3     삼화아트빌라    월세    49.02  202407   
1  19  경기도 수원시 권선구 세류동  1158-52  1158  52  (1158-52)    월세    19.80  202407   
2  20  경기도 수원시 권선구 세류동   1158-4  1158   4        채움빌    월세    12.97  202407   
3  29   경기도 수원시 권선구 평동    48-51    48  51     미주디럭스빌    월세    16.10  202407   
4  39  경기도 성남시 수정구 신흥동     2590  2590   0       렉스빌Ⅲ    월세    36.18  202407   

   계약일  ...  월세금(만원)  층  건축년도              도로명           계약기간 계약구분 갱신요구권 사용  \
0   31  ...       15  3  1992      정조로495번길 10  202410~202610   신규        -   
1   31  ...       40  3  2013    정조로398번길 11-4  202408~202511   신규        -   
2   31  ...       40  4  2015   정조로398번길 11-22  202408~202511   신규        -   
3   31  ...       35  2  2013       평동로68번길 35  202407~202507   신규        -   
4   31  ...       65  2  2007  산성

In [6]:
df = pd.read_csv('real_estate_data/cleaned/refined_data_단독다가구_fin.csv', encoding='utf-8')# 월세금이 0인 행만 필터링
zero_rent_df = df[df['월세금(만원)'] <= 10]

pd.set_option('display.max_columns', None)  # 모든 열 출력
pd.set_option('display.width', 1000)        # 가로 폭 넓게
pd.set_option('display.max_colwidth', None) # 문자열 길이 제한 해제

# 결과 출력
print(f"월세금이 20이하인 행 개수: {len(zero_rent_df)}")
print("\n월세금이 20이하인 행 샘플:")
zero_rent_df.head(30)  # 처음 5행 출력

월세금이 20이하인 행 개수: 234

월세금이 20이하인 행 샘플:


,시군구,계약면적(㎡),계약년월,보증금(만원),월세금(만원),도로명,주택유형
9,충청북도 청주시 청원구 우암동,21.87,202407,200,10,직지대로872번길,단독다가구
1778,강원특별자치도 인제군 기린면 진동리,50.00,202407,50,10,조침령로,단독다가구
3381,경상남도 김해시 진영읍 신용리,20.00,202407,50,10,진영로,단독다가구
3559,강원특별자치도 원주시 중앙동,19.80,202407,0,10,NaN,단독다가구
3827,충청북도 청주시 흥덕구 봉명동,20.00,202407,50,9,천석로35번길,단독다가구
4222,경기도 수원시 장안구 정자동,30.00,202407,0,4,정자로144번길,단독다가구
4682,강원특별자치도 강릉시 옥천동,49.00,202407,50,10,옥가로,단독다가구
5398,충청북도 청주시 서원구 개신동,13.20,202407,300,10,모충로19번길,단독다가구
5511,경기도 시흥시 정왕동,19.00,202407,0,5,오이도2길,단독다가구
5812,경상남도 합천군 합천읍 장계리,42.39,202407,114,5,NaN,단독다가구


In [ ]:
df = pd.read_csv('real_estate_data/cleaned/cleaned_merged_오피스텔_fin.csv', encoding='utf-8')# 월세금이 0인 행만 필터링
df[df['월세금(만원)'] == 1300]

C:\Users\Playdata\AppData\Local\Temp\ipykernel_23040\1833545919.py:1: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('real_estate_data/cleaned/cleaned_merged_오피스텔_fin.csv', encoding='utf-8')# 월세금이 0인 행만 필터링


,NO,시군구,번지,본번,부번,단지명,전월세구분,전용면적(㎡),계약년월,계약일,보증금(만원),월세금(만원),층,건축년도,도로명,계약기간,계약구분,갱신요구권 사용,종전계약 보증금(만원),종전계약 월세(만원)
35608,20521,경기도 고양시 덕양구 동산동,368-2,368,2,e편한세상 시티 삼송2차,월세,57.92,202501,4,"3,000",1300,9,2019.0,덕수천2로 150,202502~202702,신규,-,NaN,NaN


In [7]:
rent_df = df[df['월세금(만원)'] <= 10]
real_df = rent_df[rent_df['보증금(만원)'] < 500]
print(f"월세금이 10이하고, 보증금이 500 미만인 행 개수: {len(real_df)}")
real_df

월세금이 10이하고, 보증금이 500 미만인 행 개수: 234


,시군구,계약면적(㎡),계약년월,보증금(만원),월세금(만원),도로명,주택유형
9,충청북도 청주시 청원구 우암동,21.87,202407,200,10,직지대로872번길,단독다가구
1778,강원특별자치도 인제군 기린면 진동리,50.00,202407,50,10,조침령로,단독다가구
3381,경상남도 김해시 진영읍 신용리,20.00,202407,50,10,진영로,단독다가구
3559,강원특별자치도 원주시 중앙동,19.80,202407,0,10,NaN,단독다가구
3827,충청북도 청주시 흥덕구 봉명동,20.00,202407,50,9,천석로35번길,단독다가구
...,...,...,...,...,...,...,...
139353,경상북도 구미시 형곡동,35.56,202506,100,10,형곡로8길,단독다가구
140416,충청남도 보령시 내항동,15.00,202506,50,10,왕대산길,단독다가구
141326,경기도 하남시 덕풍동,12.10,202506,300,10,덕풍공원로87번길,단독다가구
142596,전라남도 장흥군 장흥읍 건산리,10.00,202507,0,10,장흥대로,단독다가구


In [8]:
import numpy as np
list1=sorted(rent_df['보증금(만원)'].unique())
print(np.array(list1).astype(int))

[  0   1   5  10  20  30  50  60 100 114 120 129 149 150 168 185 186 200
 237 238 239 274 293 298 299 300 302 400 401 405 442 456 468]


In [25]:
df1 = df[df['보증금(만원)'] >= 1000]
list2=sorted(df1['월세금(만원)'].unique())
print(np.array(list2).astype(int))

[ 11  12  13  14  15  16  17  18  19  20  21  22  23  24  25  26  27  28
  29  30  31  32  33  34  35  36  37  38  39  40  41  42  43  44  45  46
  47  48  49  50  51  52  53  54  55  56  57  58  59  60  61  62  63  64
  65  66  67  68  69  70  71  72  73  74  75  76  77  78  79  80  81  82
  83  84  85  86  87  88  89  90  91  92  93  94  95  96  97  98  99 100
 101 102 103 104 105 106 108 110 111 112 113 114 115 116 117 118 120 121
 122 123 125 126 128 130 132 133 135 136 137 140 145 147 150 154 156 160
 163 167 170 180 200 208 300]


In [26]:
df = pd.read_csv('real_estate_data/cleaned/cleaned_merged_단독다가구_fin.csv', encoding='utf-8')# 월세금이 0인 행만 필터링
zero_rent_df = df[df['월세금(만원)'] <= 10]
real_df = zero_rent_df[(zero_rent_df['보증금(만원)'] < 500)]

pd.set_option('display.max_columns', None)  # 모든 열 출력
pd.set_option('display.width', 1000)        # 가로 폭 넓게
pd.set_option('display.max_colwidth', None) # 문자열 길이 제한 해제

# 결과 출력
print(f"월세 10이하인 행 개수: {len(zero_rent_df)}")
print(f"월세 10이하, 보증금 2000이상인 행 개수: {len(real_df)}")
print("\n월세금이 30이하인 행 샘플:")
real_df  # 처음 5행 출력

월세 10이하인 행 개수: 3158
월세 10이하, 보증금 2000이상인 행 개수: 234

월세금이 30이하인 행 샘플:


,NO,시군구,번지,도로조건,계약면적(㎡),전월세구분,계약년월,계약일,보증금(만원),월세금(만원),건축년도,도로명,계약기간,계약구분,갱신요구권 사용,종전계약 보증금(만원),종전계약 월세(만원),주택유형
9,31,충청북도 청주시 청원구 우암동,3**,8m미만,21.87,월세,202407,31,200,10,2020.0,직지대로872번길,202407~202609,신규,-,NaN,NaN,단독다가구
1961,5973,강원특별자치도 인제군 기린면 진동리,1*,8m미만,50.00,월세,202407,26,50,10,2014.0,조침령로,202407~202707,신규,-,NaN,NaN,단독다가구
3722,11675,경상남도 김해시 진영읍 신용리,5**,8m미만,20.00,월세,202407,22,50,10,1992.0,진영로,202408~202608,신규,-,NaN,NaN,단독다가구
3915,12053,강원특별자치도 원주시 중앙동,2**,-,19.80,월세,202407,22,0,10,NaN,NaN,202407~202410,신규,-,NaN,NaN,단독다가구
4202,12701,충청북도 청주시 흥덕구 봉명동,1***,12m미만,20.00,월세,202407,20,50,9,1986.0,천석로35번길,202407~202507,신규,-,NaN,NaN,단독다가구
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151554,26784,경상북도 구미시 형곡동,1**,12m미만,35.56,월세,202506,5,100,10,1991.0,형곡로8길,202506~202512,신규,-,NaN,NaN,단독다가구
152686,30193,충청남도 보령시 내항동,7**,8m미만,15.00,월세,202506,3,50,10,NaN,왕대산길,202506~202706,신규,-,NaN,NaN,단독다가구
153651,32257,경기도 하남시 덕풍동,4**,8m미만,12.10,월세,202506,1,300,10,1990.0,덕풍공원로87번길,202506~202606,신규,-,NaN,NaN,단독다가구
155043,3612,전라남도 장흥군 장흥읍 건산리,7**,8m미만,10.00,월세,202507,1,0,10,NaN,장흥대로,202507~202707,신규,-,NaN,NaN,단독다가구


In [9]:
# 파일 읽기
df = pd.read_csv('real_estate_data/cleaned/cleaned_merged_단독다가구_fin.csv', encoding="utf-8-sig")

# 조건에 맞는 행 제거
filtered_df = df[~((df["보증금(만원)"] >= 500) & (df["월세금(만원)"] <= 10))]

# 필요한 컬럼 추출
result_df = filtered_df[["시군구", "계약면적(㎡)", "계약년월", "보증금(만원)", "월세금(만원)", "도로명", "주택유형"]]

# 결과 저장
result_df.to_csv("real_estate_data/cleaned/refined_data.csv", index=False, encoding="utf-8-sig")

In [28]:
# 파일 읽기
df = pd.read_csv('real_estate_data/cleaned/refined_data.csv', encoding="utf-8-sig")

# 조건에 맞는 행 제거
filtered_df = df[(df["보증금(만원)"] <= 3000)]
# 결과 저장
filtered_df.to_csv("real_estate_data/cleaned/refined_data_단독다가구_fin.csv", index=False, encoding="utf-8-sig")

In [2]:
import pandas as pd
df = pd.read_csv('real_estate_data/cleaned/refined_data_단독다가구_fin.csv', encoding="utf-8-sig")
df_sorted = df.sort_values(by=['계약년월', '시군구'])
df_sorted.head(100)

,시군구,계약면적(㎡),계약년월,보증금(만원),월세금(만원),도로명,주택유형
222,강원특별자치도 강릉시 교동,33.30,202407,200,40,경포로,단독다가구
798,강원특별자치도 강릉시 교동,29.70,202407,200,40,율곡초교길11번길,단독다가구
1552,강원특별자치도 강릉시 교동,40.79,202407,500,45,옛강일길38번길,단독다가구
1553,강원특별자치도 강릉시 교동,40.79,202407,500,45,옛강일길38번길,단독다가구
1736,강원특별자치도 강릉시 교동,56.00,202407,500,62,화부산로111번길,단독다가구
...,...,...,...,...,...,...,...
1555,강원특별자치도 동해시 천곡동,33.00,202407,300,40,감추5길,단독다가구
1556,강원특별자치도 동해시 천곡동,29.75,202407,200,29,샘실3길,단독다가구
2179,강원특별자치도 동해시 천곡동,21.48,202407,200,30,감추3길,단독다가구
2190,강원특별자치도 동해시 천곡동,45.00,202407,2000,28,동굴4길,단독다가구


In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 데이터 로드
df = pd.read_csv('real_estate_data/cleaned/refined_data_단독다가구_fin.csv', encoding='utf-8')

# 시군구별로 그룹화하여 보증금과 월세금의 평균 계산
grouped = df.groupby('시군구').agg({
    '보증금(만원)': 'mean',
    '월세금(만원)': 'mean',
    '계약년월': 'count'
}).rename(columns={'계약년월': '총_건수'}).reset_index()

# 시군구 내에서 계약년월별 건수 계산
contract_count = df.groupby(['시군구', '계약년월']).size().reset_index(name='계약년월별_건수')

# 결과 출력
print("시군구별 평균 보증금과 월세금:")
grouped
# print("\n시군구 내 계약년월별 건수:")
# contract_count)

시군구별 평균 보증금과 월세금:


,시군구,보증금(만원),월세금(만원),총_건수
0,강원특별자치도 강릉시 강동면 상시동리,1000.000000,70.000000,1
1,강원특별자치도 강릉시 견소동,500.000000,45.000000,1
2,강원특별자치도 강릉시 교동,341.908752,43.532588,537
3,강원특별자치도 강릉시 구정면 제비리,1000.000000,60.000000,1
4,강원특별자치도 강릉시 난곡동,1666.666667,64.222222,9
...,...,...,...,...
3126,충청북도 충주시 주덕읍 화곡리,450.000000,46.750000,4
3127,충청북도 충주시 중앙탑면 용전리,400.000000,40.899225,129
3128,충청북도 충주시 지현동,500.000000,35.000000,4
3129,충청북도 충주시 칠금동,331.666667,34.966667,30


In [4]:
df[df['시군구'].str.contains('강릉시')]

,시군구,계약면적(㎡),계약년월,보증금(만원),월세금(만원),도로명,주택유형
78,강원특별자치도 강릉시 내곡동,26.44,202407,30,35,범일로,단독다가구
79,강원특별자치도 강릉시 내곡동,26.44,202407,30,37,범일로,단독다가구
86,강원특별자치도 강릉시 지변동,29.00,202407,500,52,지변길18번길,단독다가구
222,강원특별자치도 강릉시 교동,33.30,202407,200,40,경포로,단독다가구
606,강원특별자치도 강릉시 옥천동,59.60,202407,500,50,옥천로10번길,단독다가구
...,...,...,...,...,...,...,...
142507,강원특별자치도 강릉시 교동,33.00,202507,300,50,율곡초교길11번길,단독다가구
142579,강원특별자치도 강릉시 교동,26.00,202507,500,43,하슬라로,단독다가구
142618,강원특별자치도 강릉시 지변동,19.60,202507,50,33,경포로15번길,단독다가구
142650,강원특별자치도 강릉시 지변동,21.38,202507,200,50,지변길,단독다가구


In [15]:
df[df['시군구'].str.contains('강릉시')]['보증금(만원)'].mean()

np.float64(402.99236641221376)

In [16]:
df[df['시군구'].str.contains('강릉시')]['월세금(만원)'].mean()

np.float64(43.451908396946564)

In [17]:
df[df['시군구'].str.contains('충주시')]['보증금(만원)'].mean()

np.float64(497.6393659180978)

In [18]:
df[df['시군구'].str.contains('충주시')]['월세금(만원)'].mean()

np.float64(40.11889035667107)

In [5]:
import pandas as pd

# CSV 읽기
df = pd.read_csv('real_estate_data/cleaned/refined_data_단독다가구_fin.csv', encoding='utf-8')

# ▶ 시군구 컬럼에서 두 번째 단어 (시/군/구) 추출
def extract_city_name(addr):
    if isinstance(addr, str):
        if "세종특별자치시" in addr:
            return "세종특별자치시"
        else:
            parts = addr.split()
            if len(parts) > 1:
                return parts[1]  # 예: '강릉시', '수원시' 등
    return None

df['시군'] = df['시군구'].apply(extract_city_name)

# df['시군'] = df['시군구'].apply(lambda x: str(x).split()[1] if isinstance(x, str) and len(x.split()) > 1 else None)

# ▶ 보증금과 월세를 숫자형으로 변환 (NaN 처리 포함)
df['보증금(만원)'] = pd.to_numeric(df['보증금(만원)'], errors='coerce')
df['월세금(만원)'] = pd.to_numeric(df['월세금(만원)'], errors='coerce')

# ▶ 시군별 평균 계산
avg_df = df.groupby('시군')[['보증금(만원)', '월세금(만원)']].mean().reset_index()

# ▶ 결과 확인
print(avg_df.sort_values(by='보증금(만원)'))

      시군      보증금(만원)    월세금(만원)
67   신안군   125.000000  20.000000
84   영암군   144.444444  28.000000
78   여수시   244.031915  42.351064
124  철원군   250.000000  30.800000
17   광양시   250.837989  37.240223
..   ...          ...        ...
106  인제군  1091.463415  36.585366
59   성남시  1113.437535  48.855169
140  하남시  1157.034384  48.151862
102  의왕시  1291.638225  50.955631
15   과천시  1489.430894  54.109756

[153 rows x 3 columns]


In [6]:
    # 1. 시도, 시군 모두 추출
def extract_location_parts(addr):
    if isinstance(addr, str):
        if "세종특별자치시" in addr:
            return ("세종특별자치시", "")
        else:
            parts = addr.split()
            if len(parts) > 1:
                return (parts[0], parts[1])
    return (None, None)

df[['시도', '시군']] = df['시군구'].apply(lambda x: pd.Series(extract_location_parts(x)))

# 2. 숫자 변환
df['보증금(만원)'] = pd.to_numeric(df['보증금(만원)'], errors='coerce')
df['월세금(만원)'] = pd.to_numeric(df['월세금(만원)'], errors='coerce')

# 3. 시군 단위로 평균 계산 (행정구 기준 유지)
avg_df = df.groupby('시군')[['보증금(만원)', '월세금(만원)']].mean().reset_index()

# 4. 시도 정보 결합해서 표시용 컬럼 추가
# (시군 -> 시도 mapping 필요)
sido_mapping = df.drop_duplicates('시군')[['시군', '시도']].set_index('시군').to_dict()['시도']
avg_df['시도'] = avg_df['시군'].map(sido_mapping)
avg_df['지역명'] = avg_df['시도'] + " " + avg_df['시군']

# 5. 보기 좋게 정렬
avg_df = avg_df.sort_values(by='보증금(만원)', ascending=False)

# 6. 출력
# print(avg_df[['지역명', '보증금(만원)', '월세금(만원)']].tail(30))  # 상위 10개만 출력
print(avg_df[['지역명', '보증금(만원)', '월세금(만원)']])

# 특정 열만 선택하여 저장
# avg_df[['지역명', '보증금(만원)', '월세금(만원)']].to_csv('평균_보증금_월세금.csv', index=False, encoding='utf-8-sig')
avg_df[['지역명']].to_csv('지역명.csv', index=False, encoding='utf-8-sig')

             지역명      보증금(만원)    월세금(만원)
16       경기도 과천시  1489.430894  54.109756
102      경기도 의왕시  1291.638225  50.955631
140      경기도 하남시  1157.034384  48.151862
60       경기도 성남시  1113.437535  48.855169
106  강원특별자치도 인제군  1091.463415  36.585366
..           ...          ...        ...
18      전라남도 광양시   250.837989  37.240223
124  강원특별자치도 철원군   250.000000  30.800000
78      전라남도 여수시   244.031915  42.351064
84      전라남도 영암군   144.444444  28.000000
67      전라남도 신안군   125.000000  20.000000

[153 rows x 3 columns]


In [29]:
avg_df[avg_df['지역명'].str.contains('세종', na=False)]

,시군,보증금(만원),월세금(만원),시도,지역명
0,,357.011966,39.006838,세종특별자치시,세종특별자치시


In [7]:
from sklearn.cluster import KMeans
import numpy as np

# 데이터 준비
X = df[['보증금(만원)', '월세금(만원)']].values

# 3개의 그룹으로 클러스터링
kmeans = KMeans(n_clusters=3, random_state=0, n_init=10)
df['Cluster'] = kmeans.fit_predict(X)

# 클러스터별 평균을 보고 등급 매기기 (클러스터 0,1,2를 상,중,하로 매핑)
cluster_means = df.groupby('Cluster')[['보증금(만원)', '월세금(만원)']].mean().sort_values('월세금(만원)', ascending=False)
cluster_mapping = {cluster_means.index[i]: grade for i, grade in enumerate(['상', '중', '하'])}
df['클러스터등급'] = df['Cluster'].map(cluster_mapping)

In [8]:
import pandas as pd

# 1. 데이터 불러오기
df = pd.read_csv("평균_보증금_월세금.csv")

# 2. 전월세 변환율 적용 (5% 기준) - 보증금과 월세를 함께 고려
df['종합월부담'] = (df['보증금(만원)'] * 0.05 / 12) + df['월세금(만원)']
df['종합월부담'] = df['종합월부담'].round(2)  # 소수점 2자리까지 반올림

# 3. 종합월부담 기준으로 등급 분류
mean_total = df['종합월부담'].mean()
std_total = df['종합월부담'].std()

upper_bound = mean_total + 0.5 * std_total
lower_bound = mean_total - 0.5 * std_total

def classify_total(value):
    if value >= upper_bound:
        return "상"
    elif value <= lower_bound:
        return "하"
    else:
        return "중"

df['종합부담등급'] = df['종합월부담'].apply(classify_total)

# 4. 월세만 따로 등급 분류 (비교용)
mean_rent = df['월세금(만원)'].mean()
std_rent = df['월세금(만원)'].std()

def classify_rent(value):
    if value >= mean_rent + 0.5 * std_rent:
        return "상"
    elif value <= mean_rent - 0.5 * std_rent:
        return "하"
    else:
        return "중"

df['월세등급'] = df['월세금(만원)'].apply(classify_rent)

# 5. 결과 출력
print("=" * 80)
print("📊 월세 및 종합 주거비 부담 분석 결과")
print("=" * 80)

print(f"\n📈 기준값 정보:")
print(f"- 월세 평균: {mean_rent:.2f}만원, 표준편차: {std_rent:.2f}만원")
print(f"- 종합월부담 평균: {mean_total:.2f}만원, 표준편차: {std_total:.2f}만원")
print(f"- 종합월부담 구간: 하({lower_bound:.2f} 이하) | 중({lower_bound:.2f} ~ {upper_bound:.2f}) | 상({upper_bound:.2f} 이상)")

print(f"\n🏆 등급별 지역 수 (종합부담기준):")
print(df['종합부담등급'].value_counts().sort_index())

print(f"\n🏆 등급별 지역 수 (월세only기준):")
print(df['월세등급'].value_counts().sort_index())

print(f"\n🔝 종합 부담 상위 10개 지역:")
top10_total = df.nlargest(10, '종합월부담')[['지역명', '보증금(만원)', '월세금(만원)', '종합월부담', '종합부담등급']]
print(top10_total.to_string(index=False))

print(f"\n🔝 월세 only 상위 10개 지역:")
top10_rent = df.nlargest(10, '월세금(만원)')[['지역명', '보증금(만원)', '월세금(만원)', '월세등급']]
print(top10_rent.to_string(index=False))

print(f"\n🔽 종합 부담 하위 10개 지역:")
bottom10_total = df.nsmallest(10, '종합월부담')[['지역명', '보증금(만원)', '월세금(만원)', '종합월부담', '종합부담등급']]
print(bottom10_total.to_string(index=False))

# 6. CSV 파일로 저장
output_filename = "평균_보증금_월세금_등급포함.csv"
df.to_csv(output_filename, index=False, encoding='utf-8-sig')

print(f"\n💾 분석 결과가 '{output_filename}' 파일로 저장되었습니다.")
print("=" * 80)

📊 월세 및 종합 주거비 부담 분석 결과

📈 기준값 정보:
- 월세 평균: 39.18만원, 표준편차: 5.73만원
- 종합월부담 평균: 41.70만원, 표준편차: 6.13만원
- 종합월부담 구간: 하(38.63 이하) | 중(38.63 ~ 44.76) | 상(44.76 이상)

🏆 등급별 지역 수 (종합부담기준):
종합부담등급
상    39
중    68
하    46
Name: count, dtype: int64

🏆 등급별 지역 수 (월세only기준):
월세등급
상    39
중    68
하    46
Name: count, dtype: int64

🔝 종합 부담 상위 10개 지역:
        지역명     보증금(만원)   월세금(만원)  종합월부담 종합부담등급
    경기도 과천시 1489.430894 54.109756  60.32      상
    경기도 평택시  493.667310 54.350208  56.41      상
제주특별자치도 제주시  326.652715 55.002767  56.36      상
    경기도 의왕시 1291.638225 50.955631  56.34      상
    경기도 파주시  893.593079 51.612768  55.34      상
    경기도 성남시 1113.437535 48.855169  53.49      상
    경기도 화성시  831.598298 49.880430  53.35      상
    경기도 하남시 1157.034384 48.151862  52.97      상
   경기도 남양주시  974.538917 48.834602  52.90      상
    경기도 용인시  871.875038 49.123117  52.76      상

🔝 월세 only 상위 10개 지역:
         지역명     보증금(만원)   월세금(만원) 월세등급
 제주특별자치도 제주시  326.652715 55.002767    상
     경기도 평택시  493.667310 54.350208  